# AgroDoctor public-data research trial

Fine-tune MobileNetV3 Small with the original PlantVillage split and additional public field photos. The Rachin public tomato cohort stays offline for a separate exploratory check. Keep the Kaggle notebook and dataset private.


## 1. Choose training settings

Use seed 2026, 10 epochs and batch size 64 for this trial. If a GPU reports that it is out of memory, reduce batch size and keep that value with the results.

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import sys

INPUT_MOUNT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
BUNDLE_ROOT = None  # Set to a Path only when you attached multiple AgroDoctor input bundles.
EPOCHS = 10
UNFREEZE_BLOCKS = 2
LEARNING_RATE = 0.0005
BATCH_SIZE = 64
SEED = 2026

## 2. Find and verify the input

This cell checks the packaged code, dataset manifest and official pretrained weights before training.

In [ ]:
def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


if BUNDLE_ROOT is None:
    markers = sorted(INPUT_MOUNT.rglob("agrodoctor-input.json"))
    if len(markers) != 1:
        raise RuntimeError(
            "Attach exactly one private AgroDoctor training dataset. "
            "If you attached multiple AgroDoctor bundles, set BUNDLE_ROOT to the correct dataset folder."
        )
    bundle_root = markers[0].parent
else:
    bundle_root = Path(BUNDLE_ROOT)
    markers = [bundle_root / "agrodoctor-input.json"]

bundle = json.loads(markers[0].read_text(encoding="utf-8"))
if bundle["bundle_version"] != 1:
    raise RuntimeError("This notebook expects AgroDoctor input bundle version 1.")
for relative_path, expected in bundle["files"].items():
    artifact = bundle_root / relative_path
    if sha256_file(artifact) != expected:
        raise RuntimeError(f"Input checksum mismatch: {relative_path}")

data_root = bundle_root / "data"
trainer = bundle_root / "code" / "train_kaggle.py"
weights = bundle_root / "weights" / "mobilenet_v3_small-047dcff4.pth"
command = [
    sys.executable, "-u", str(trainer),
    "--root", str(data_root),
    "--manifest", str(data_root / "manifest.csv"),
    "--sources", str(data_root / "sources.json"),
]
if bundle.get("development_only") is True:
    command.append("--skip-test")
subprocess.run([*command, "--validate-only"], check=True)

## 3. Train on one GPU

The first epoch gives a useful estimate of the remaining runtime. The trainer chooses the checkpoint and uncertainty thresholds using validation only. When you attach the PlantVillage baseline bundle, the controlled test is evaluated afterward.

In [ ]:
from datetime import datetime, timezone
import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU under Kaggle Settings → Accelerator before running this cell.")

gpu = torch.cuda.get_device_properties(0)
print(f"{gpu.name}, {gpu.total_memory / (1024 ** 3):.1f} GiB VRAM")
print(f"PyTorch {torch.__version__} · TorchVision {torchvision.__version__}")

run_dir = WORKING / "agrodoctor" / datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
subprocess.run([
    *command,
    "--output", str(run_dir),
    "--device", "cuda",
    "--pretrained-weights", str(weights),
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--seed", str(SEED),
    "--unfreeze-blocks", str(UNFREEZE_BLOCKS),
    "--learning-rate", str(LEARNING_RATE),
], check=True)

## 4. Save the model and evaluation

Use **Save Version → Save & Run All** when you are ready for the final run. It executes the notebook from a fresh session. If you already trained interactively, download `agrodoctor-results.zip` from the Output panel before closing the session to keep that run.

The ZIP contains the inference model, checkpoint, metadata, class labels, metrics and training records. For field development, download the frozen candidate and use the local field evaluator on the offline holdout.

In [ ]:
archive = Path(shutil.make_archive(str(WORKING / "agrodoctor-results"), "zip", root_dir=run_dir))
print(f"Download from the Output panel: {archive.name}")
print("Included:")
for output_file in sorted(run_dir.iterdir()):
    print(output_file.name)